# bd3.ipynb

`sample` 폴더 안의 모든 영상에 대해 다음을 한 번에 수행합니다.

1. YOLO 객체 추출 결과를 `cf2/*.json`으로 저장
2. cf2 JSON을 기반으로 bbox pseudo-tracking 생성
3. 싸움 후보 탐지
4. 스토킹 후보 탐지
5. 최종적으로 `fight / stalking / other` 중 하나로 분류
6. 영상별 prediction JSON, 전체 summary JSON, 이벤트 이미지/클립 저장

> 주의: 이 노트북은 pose/keypoint 모델 없이 `person bbox`만으로 손/발 영역과 추적을 근사합니다. 따라서 최종 결과는 확정 판정이 아니라 **후보 상황 분류**입니다.

In [1]:
# ==============================
# 0. 기본 설정
# ==============================
from pathlib import Path
import json
import math
import cv2
import numpy as np
from itertools import combinations, product
from collections import defaultdict, Counter
from tqdm import tqdm

try:
    from ultralytics import YOLO
except ImportError as e:
    raise ImportError("ultralytics가 설치되어 있지 않습니다. 먼저 pip install ultralytics 를 실행하세요.") from e

BASE_DIR = Path(r"C:\Users\우성한")

SAMPLE_DIR = BASE_DIR / "sample"
CF2_DIR = BASE_DIR / "cf2"
OUTPUT_DIR = BASE_DIR / "outputs" / "prediction" / "bd3"
EVENT_FRAME_DIR = OUTPUT_DIR / "event_frames"
EVENT_CLIP_DIR = OUTPUT_DIR / "event_clips"

CF2_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVENT_FRAME_DIR.mkdir(parents=True, exist_ok=True)
EVENT_CLIP_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = [".mp4", ".avi", ".mov", ".mkv", ".wmv", ".m4v"]

# 객체추출 옵션
YOLO_MODEL_NAME = "yolov8n.pt"  # 느리면 n, 조금 더 정확히는 s
FRAME_STRIDE = 10                # 10프레임마다 1번 탐지
CONF_TH = 0.25
OVERWRITE_CF2 = False            # True면 기존 cf2 JSON도 다시 생성

# pseudo-tracking 옵션
MIN_TRACK_LENGTH = 5
TRACK_IOU_TH = 0.10
TRACK_CENTER_DIST_TH = 180
TRACK_MAX_MISSING = 2

# 싸움 탐지 옵션
FIGHT_NEAR_TH = 90               # 손/발 근사 bbox와 상대 bbox 거리
FIGHT_IOU_TH = 0.01              # 손/발 근사 bbox가 상대 bbox와 조금이라도 겹치면 접촉 후보
FIGHT_WINDOW_SEC = 3.0
FIGHT_MIN_CONTACT_COUNT = 2      # 사용자 요구: 2회 이상 반복
FIGHT_MIN_ACTIVE_SPEED = 5.0

# 스토킹 탐지 옵션
STALKING_MIN_DIST = 60
STALKING_MAX_DIST = 420
STALKING_DIR_SIM_TH = 0.55
STALKING_BEHIND_DOT_TH = 10      # target 진행방향 기준 follower가 뒤에 있어야 함
STALKING_WINDOW_SEC = 6.0
STALKING_MIN_RATIO = 0.45
STALKING_DISTANCE_STD_TH = 140

# 최종 분류 우선순위
# fight와 stalking이 동시에 나오면 fight 우선. 필요하면 바꿔도 됨.
PRIORITY = ["fight", "stalking", "other"]

print("SAMPLE_DIR:", SAMPLE_DIR)
print("CF2_DIR:", CF2_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

SAMPLE_DIR: C:\Users\우성한\sample
CF2_DIR: C:\Users\우성한\cf2
OUTPUT_DIR: C:\Users\우성한\outputs\prediction\bd3


In [2]:
# ==============================
# 1. sample 전체 영상 목록 수집
# ==============================
def collect_videos(sample_dir):
    videos = []
    for ext in VIDEO_EXTS:
        videos.extend(sample_dir.rglob(f"*{ext}"))
    return sorted(set(videos))

video_paths = collect_videos(SAMPLE_DIR)

print("sample 영상 개수:", len(video_paths))
for p in video_paths:
    print("-", p)

sample 영상 개수: 0


In [3]:
# ==============================
# 2. 공통 유틸
# ==============================
def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def safe_stem_from_video(video_path):
    """하위 폴더 영상까지 이름 충돌 없이 cf2 JSON명을 만들기 위한 함수."""
    rel = Path(video_path).relative_to(SAMPLE_DIR)
    safe = str(rel).replace("\\", "__").replace("/", "__")
    return Path(safe).with_suffix("").name


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return [(x1 + x2) / 2, (y1 + y2) / 2]


def bbox_area(b):
    x1, y1, x2, y2 = b
    return max(0, x2 - x1) * max(0, y2 - y1)


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = bbox_area([ix1, iy1, ix2, iy2])
    union = bbox_area(a) + bbox_area(b) - inter
    if union <= 0:
        return 0.0
    return inter / union


def point_dist(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def bbox_distance(a, b):
    """두 bbox가 겹치면 0, 아니면 최소 x/y gap 기반 거리."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    dx = max(bx1 - ax2, ax1 - bx2, 0)
    dy = max(by1 - ay2, ay1 - by2, 0)
    return math.sqrt(dx * dx + dy * dy)


def vec(a, b):
    return [b[0] - a[0], b[1] - a[1]]


def norm(v):
    return math.sqrt(v[0] ** 2 + v[1] ** 2)


def cosine(v1, v2):
    n1, n2 = norm(v1), norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return (v1[0] * v2[0] + v1[1] * v2[1]) / (n1 * n2)


def mean(xs):
    return sum(xs) / len(xs) if xs else 0.0


def std(xs):
    if len(xs) < 2:
        return 0.0
    m = mean(xs)
    return math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs))

In [4]:
# ==============================
# 3. sample 전체 영상 → cf2 JSON 객체추출
# ==============================
model = YOLO(YOLO_MODEL_NAME)
print("YOLO model loaded:", YOLO_MODEL_NAME)


def extract_objects_from_video(video_path, model, frame_stride=10, conf_th=0.25):
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frames = []
    class_total = Counter()
    frame_id = 0
    sampled_count = 0

    pbar = tqdm(total=total_frames, desc=f"extract {video_path.name}", leave=False)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_stride == 0:
            time_sec = frame_id / fps if fps and fps > 0 else 0.0
            result = model(frame, conf=conf_th, verbose=False)[0]

            objects = []
            classes = Counter()

            for box in result.boxes:
                cls_id = int(box.cls[0])
                class_name = model.names[cls_id]
                conf = float(box.conf[0])
                x1, y1, x2, y2 = box.xyxy[0].tolist()

                bbox = [round(float(x1), 2), round(float(y1), 2), round(float(x2), 2), round(float(y2), 2)]

                objects.append({
                    "class": class_name,
                    "class_name": class_name,
                    "confidence": round(conf, 4),
                    "bbox": bbox
                })
                classes[class_name] += 1
                class_total[class_name] += 1

            frames.append({
                "frame_id": int(frame_id),
                "time_sec": round(float(time_sec), 3),
                "object_count": len(objects),
                "classes": dict(classes),
                "objects": objects
            })
            sampled_count += 1

        frame_id += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    return {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "summary": {
            "fps": fps,
            "width": width,
            "height": height,
            "total_frames": total_frames,
            "sampled_frames": sampled_count,
            "frame_stride": frame_stride,
            "class_total": dict(class_total)
        },
        "frames": frames
    }


def ensure_cf2_for_all_videos(video_paths):
    rows = []
    for video_path in video_paths:
        stem = safe_stem_from_video(video_path)
        output_json = CF2_DIR / f"{stem}.json"

        row = {
            "video_path": str(video_path),
            "cf2_json": str(output_json),
            "status": None,
            "error": None
        }

        if output_json.exists() and not OVERWRITE_CF2:
            row["status"] = "skipped_exists"
            rows.append(row)
            print(f"[CF2 SKIP] {output_json.name}")
            continue

        try:
            print(f"\n[CF2 생성] {video_path.name}")
            data = extract_objects_from_video(video_path, model, FRAME_STRIDE, CONF_TH)
            data["cf2_source_video"] = str(video_path)
            save_json(data, output_json)
            row["status"] = "created"
            print(f"[CF2 저장] {output_json}")
        except Exception as e:
            row["status"] = "failed"
            row["error"] = repr(e)
            print(f"[CF2 실패] {video_path.name}: {e}")

        rows.append(row)

    return rows

cf2_generation_log = ensure_cf2_for_all_videos(video_paths)
save_json(cf2_generation_log, OUTPUT_DIR / "bd3_cf2_generation_log.json")

print("\n================================")
print("CF2 생성/확인 완료")
print("전체 영상:", len(video_paths))
print("생성:", sum(1 for r in cf2_generation_log if r["status"] == "created"))
print("기존 사용:", sum(1 for r in cf2_generation_log if r["status"] == "skipped_exists"))
print("실패:", sum(1 for r in cf2_generation_log if r["status"] == "failed"))
print("================================")

YOLO model loaded: yolov8n.pt

CF2 생성/확인 완료
전체 영상: 0
생성: 0
기존 사용: 0
실패: 0


In [5]:
# ==============================
# 4. cf2 JSON → person pseudo-tracking
# ==============================
def get_person_detections_from_cf2(cf2_data):
    detections_by_frame = []
    for frame in cf2_data.get("frames", []):
        frame_id = int(frame.get("frame_id", 0))
        time_sec = float(frame.get("time_sec", 0.0))
        persons = []

        for obj in frame.get("objects", []):
            cls = obj.get("class") or obj.get("class_name")
            if cls not in ["person", "human"]:
                continue
            bbox = [float(x) for x in obj.get("bbox", [])]
            if len(bbox) != 4:
                continue
            persons.append({
                "frame_id": frame_id,
                "time_sec": time_sec,
                "bbox": bbox,
                "center": bbox_center(bbox),
                "confidence": obj.get("confidence")
            })

        detections_by_frame.append({
            "frame_id": frame_id,
            "time_sec": time_sec,
            "persons": persons
        })
    return detections_by_frame


def assign_pseudo_tracks(cf2_data):
    frames = get_person_detections_from_cf2(cf2_data)
    active_tracks = {}
    all_tracks = defaultdict(list)
    next_tid = 1

    for fr in frames:
        frame_id = fr["frame_id"]
        persons = fr["persons"]
        assigned_det = set()
        assigned_tracks = set()

        # 후보 매칭: 기존 active track 마지막 bbox와 현재 detection
        candidates = []
        for tid, last in active_tracks.items():
            miss = last.get("missing", 0)
            if miss > TRACK_MAX_MISSING:
                continue
            for di, det in enumerate(persons):
                iou = bbox_iou(last["bbox"], det["bbox"])
                cd = point_dist(last["center"], det["center"])
                ok = iou >= TRACK_IOU_TH or cd <= TRACK_CENTER_DIST_TH
                if ok:
                    # 점수가 높을수록 우선
                    score = iou * 1000 - cd
                    candidates.append((score, tid, di, iou, cd))

        candidates.sort(reverse=True)

        for score, tid, di, iou, cd in candidates:
            if tid in assigned_tracks or di in assigned_det:
                continue
            det = persons[di].copy()
            det["track_id"] = str(tid)
            det["match_iou"] = round(iou, 4)
            det["match_center_dist"] = round(cd, 2)
            all_tracks[str(tid)].append(det)
            active_tracks[tid] = {**det, "missing": 0}
            assigned_tracks.add(tid)
            assigned_det.add(di)

        # 매칭 안 된 detection은 새 track
        for di, det in enumerate(persons):
            if di in assigned_det:
                continue
            tid = next_tid
            next_tid += 1
            det = det.copy()
            det["track_id"] = str(tid)
            det["match_iou"] = None
            det["match_center_dist"] = None
            all_tracks[str(tid)].append(det)
            active_tracks[tid] = {**det, "missing": 0}

        # 이번 frame에서 안 보인 active track missing 증가
        for tid in list(active_tracks.keys()):
            if tid not in assigned_tracks and str(tid) not in [str(p.get("track_id")) for p in persons]:
                active_tracks[tid]["missing"] = active_tracks[tid].get("missing", 0) + 1
            if active_tracks[tid].get("missing", 0) > TRACK_MAX_MISSING:
                active_tracks.pop(tid, None)

    cleaned = {}
    for tid, pts in all_tracks.items():
        pts = sorted(pts, key=lambda x: x["frame_id"])
        if len(pts) >= MIN_TRACK_LENGTH:
            cleaned[tid] = pts

    return cleaned


def build_frame_map(tracks):
    frame_map = defaultdict(dict)
    for tid, pts in tracks.items():
        for p in pts:
            frame_map[p["frame_id"]][tid] = p
    return frame_map


def point_by_frame(track_points):
    return {p["frame_id"]: p for p in track_points}


def prev_point(track_map, frame_id):
    prevs = [f for f in track_map.keys() if f < frame_id]
    if not prevs:
        return None
    return track_map[max(prevs)]


def track_speed(track_map, frame_id):
    cur = track_map.get(frame_id)
    prev = prev_point(track_map, frame_id)
    if cur is None or prev is None:
        return 0.0
    return point_dist(cur["center"], prev["center"])


def track_direction(track_map, frame_id):
    cur = track_map.get(frame_id)
    prev = prev_point(track_map, frame_id)
    if cur is None or prev is None:
        return [0.0, 0.0]
    return vec(prev["center"], cur["center"])


def summarize_tracks(tracks):
    out = {}
    for tid, pts in tracks.items():
        out[tid] = {
            "track_length": len(pts),
            "first_frame": pts[0]["frame_id"],
            "last_frame": pts[-1]["frame_id"],
            "first_time": pts[0]["time_sec"],
            "last_time": pts[-1]["time_sec"],
            "first_center": [round(x, 2) for x in pts[0]["center"]],
            "last_center": [round(x, 2) for x in pts[-1]["center"]],
        }
    return out

In [6]:
# ==============================
# 5. 싸움 탐지
# ==============================
def limb_candidate_boxes(person_bbox):
    """
    pose 없이 person bbox만으로 손/발 영역을 근사한다.
    - left/right hand: 상체 중간 좌우 바깥쪽
    - left/right foot: 하단 좌우
    """
    x1, y1, x2, y2 = person_bbox
    w = x2 - x1
    h = y2 - y1

    # bbox 크기 비율 기반 근사 영역
    hand_w = w * 0.28
    hand_h = h * 0.22
    foot_w = w * 0.32
    foot_h = h * 0.18

    y_hand1 = y1 + h * 0.25
    y_hand2 = y1 + h * 0.60
    y_foot1 = y1 + h * 0.78
    y_foot2 = y2

    boxes = {
        "left_hand_roi": [x1 - hand_w * 0.15, y_hand1, x1 + hand_w, y_hand2],
        "right_hand_roi": [x2 - hand_w, y_hand1, x2 + hand_w * 0.15, y_hand2],
        "left_foot_roi": [x1, y_foot1, x1 + foot_w, y_foot2],
        "right_foot_roi": [x2 - foot_w, y_foot1, x2, y_foot2],
    }
    return boxes


def contact_signal(actor, target):
    limb_boxes = limb_candidate_boxes(actor["bbox"])
    best = None

    for limb_name, limb_box in limb_boxes.items():
        iou = bbox_iou(limb_box, target["bbox"])
        d = bbox_distance(limb_box, target["bbox"])
        is_contact = (iou >= FIGHT_IOU_TH) or (d <= FIGHT_NEAR_TH)
        score = iou * 1000 - d

        cand = {
            "limb": limb_name,
            "limb_bbox": [round(v, 2) for v in limb_box],
            "target_bbox": [round(v, 2) for v in target["bbox"]],
            "iou": round(iou, 4),
            "distance": round(d, 2),
            "contact": is_contact,
            "score": score
        }
        if best is None or cand["score"] > best["score"]:
            best = cand

    return best


def detect_fight(tracks, frame_map):
    track_maps = {tid: point_by_frame(pts) for tid, pts in tracks.items()}
    pair_signals = defaultdict(list)

    for frame_id in sorted(frame_map.keys()):
        persons = frame_map[frame_id]
        if len(persons) < 2:
            continue

        for tid_a, tid_b in combinations(sorted(persons.keys()), 2):
            a = persons[tid_a]
            b = persons[tid_b]

            speed_a = track_speed(track_maps[tid_a], frame_id)
            speed_b = track_speed(track_maps[tid_b], frame_id)
            active = max(speed_a, speed_b) >= FIGHT_MIN_ACTIVE_SPEED

            a_to_b = contact_signal(a, b)
            b_to_a = contact_signal(b, a)

            contact = (a_to_b["contact"] or b_to_a["contact"]) and active
            best = a_to_b if a_to_b["score"] >= b_to_a["score"] else b_to_a

            pair_key = tuple(sorted([tid_a, tid_b]))
            pair_signals[pair_key].append({
                "frame_id": frame_id,
                "time_sec": float(a.get("time_sec", 0.0)),
                "track_pair": list(pair_key),
                "speed_a": round(speed_a, 2),
                "speed_b": round(speed_b, 2),
                "active": active,
                "contact": contact,
                "best_limb": best["limb"],
                "limb_bbox": best["limb_bbox"],
                "target_bbox": best["target_bbox"],
                "limb_target_iou": best["iou"],
                "limb_target_distance": best["distance"],
                "a_bbox": [round(x, 2) for x in a["bbox"]],
                "b_bbox": [round(x, 2) for x in b["bbox"]],
            })

    events = []
    for pair_key, signals in pair_signals.items():
        start_idx = 0
        while start_idx < len(signals):
            start_time = signals[start_idx]["time_sec"]
            window = []
            for j in range(start_idx, len(signals)):
                if signals[j]["time_sec"] - start_time <= FIGHT_WINDOW_SEC:
                    window.append(signals[j])
                else:
                    break

            contact_count = sum(1 for s in window if s["contact"])
            if contact_count >= FIGHT_MIN_CONTACT_COUNT:
                ratio = contact_count / max(1, len(window))
                events.append({
                    "event_class": "fight",
                    "detected": True,
                    "track_pair": list(pair_key),
                    "start_frame": window[0]["frame_id"],
                    "end_frame": window[-1]["frame_id"],
                    "start_time": round(window[0]["time_sec"], 3),
                    "end_time": round(window[-1]["time_sec"], 3),
                    "confidence": round(min(0.95, 0.55 + ratio * 0.4 + min(contact_count, 5) * 0.02), 3),
                    "reason": "손/발 근사 ROI가 상대 bbox와 겹치거나 가까워지는 상황이 짧은 구간에서 2회 이상 반복됨.",
                    "evidence": {
                        "contact_count": contact_count,
                        "window_count": len(window),
                        "contact_ratio": round(ratio, 3),
                        "sample_frames": [s for s in window if s["contact"]][:5]
                    }
                })
                start_idx += max(1, len(window))
            else:
                start_idx += 1

    return merge_events(events, event_class="fight")

In [7]:
# ==============================
# 6. 스토킹 탐지
# ==============================
def detect_stalking(tracks, frame_map):
    track_maps = {tid: point_by_frame(pts) for tid, pts in tracks.items()}
    directional_signals = defaultdict(list)  # (follower, target) 방향성 pair

    for frame_id in sorted(frame_map.keys()):
        persons = frame_map[frame_id]
        if len(persons) < 2:
            continue

        for follower_id, target_id in product(sorted(persons.keys()), sorted(persons.keys())):
            if follower_id == target_id:
                continue

            follower = persons[follower_id]
            target = persons[target_id]

            f_dir = track_direction(track_maps[follower_id], frame_id)
            t_dir = track_direction(track_maps[target_id], frame_id)
            sim = cosine(f_dir, t_dir)

            f_speed = track_speed(track_maps[follower_id], frame_id)
            t_speed = track_speed(track_maps[target_id], frame_id)

            d = point_dist(follower["center"], target["center"])

            # target 진행방향 기준 follower가 뒤에 있는지 검사
            # target_to_follower dot target_dir < 0 이면 follower가 target의 뒤쪽
            target_to_follower = vec(target["center"], follower["center"])
            behind_score = -(target_to_follower[0] * t_dir[0] + target_to_follower[1] * t_dir[1])
            behind_ok = behind_score >= STALKING_BEHIND_DOT_TH

            distance_ok = STALKING_MIN_DIST <= d <= STALKING_MAX_DIST
            direction_ok = sim >= STALKING_DIR_SIM_TH
            moving_ok = f_speed > 1.5 and t_speed > 1.5

            signal = distance_ok and direction_ok and moving_ok and behind_ok

            directional_signals[(follower_id, target_id)].append({
                "frame_id": frame_id,
                "time_sec": float(follower.get("time_sec", 0.0)),
                "follower_track_id": follower_id,
                "target_track_id": target_id,
                "distance": round(d, 2),
                "direction_similarity": round(sim, 3),
                "follower_speed": round(f_speed, 2),
                "target_speed": round(t_speed, 2),
                "behind_score": round(behind_score, 2),
                "signal": signal,
                "follower_bbox": [round(x, 2) for x in follower["bbox"]],
                "target_bbox": [round(x, 2) for x in target["bbox"]],
            })

    events = []
    for pair_key, signals in directional_signals.items():
        start_idx = 0
        while start_idx < len(signals):
            start_time = signals[start_idx]["time_sec"]
            window = []
            for j in range(start_idx, len(signals)):
                if signals[j]["time_sec"] - start_time <= STALKING_WINDOW_SEC:
                    window.append(signals[j])
                else:
                    break

            if len(window) < 5:
                start_idx += 1
                continue

            true_items = [s for s in window if s["signal"]]
            ratio = len(true_items) / len(window)
            distances = [s["distance"] for s in true_items]
            distance_std = std(distances)

            if ratio >= STALKING_MIN_RATIO and distance_std <= STALKING_DISTANCE_STD_TH:
                follower_id, target_id = pair_key
                events.append({
                    "event_class": "stalking",
                    "detected": True,
                    "stalking_type": "following_from_behind",
                    "follower_track_id": follower_id,
                    "target_track_id": target_id,
                    "start_frame": window[0]["frame_id"],
                    "end_frame": window[-1]["frame_id"],
                    "start_time": round(window[0]["time_sec"], 3),
                    "end_time": round(window[-1]["time_sec"], 3),
                    "confidence": round(min(0.93, 0.50 + ratio * 0.35 + max(0, 1 - distance_std / 200) * 0.08), 3),
                    "reason": "한 사람이 다른 사람의 뒤쪽에서 비슷한 방향으로 일정 거리 범위를 유지하며 이동함.",
                    "evidence": {
                        "signal_ratio": round(ratio, 3),
                        "signal_count": len(true_items),
                        "window_count": len(window),
                        "distance_std": round(distance_std, 3),
                        "sample_frames": true_items[:5]
                    }
                })
                start_idx += max(1, len(window))
            else:
                start_idx += 1

    return merge_events(events, event_class="stalking")

In [8]:
# ==============================
# 7. 이벤트 병합 / 최종 분류
# ==============================
def same_event_pair(a, b):
    if "track_pair" in a and "track_pair" in b:
        return sorted(a["track_pair"]) == sorted(b["track_pair"])
    if "follower_track_id" in a and "follower_track_id" in b:
        return (
            a.get("follower_track_id") == b.get("follower_track_id")
            and a.get("target_track_id") == b.get("target_track_id")
        )
    return False


def merge_events(events, event_class, max_gap_frames=30):
    if not events:
        return []
    events = sorted(events, key=lambda x: (x["start_frame"], x["end_frame"]))
    merged = []

    for ev in events:
        if not merged:
            merged.append(ev)
            continue
        last = merged[-1]
        if (
            last.get("event_class") == ev.get("event_class") == event_class
            and same_event_pair(last, ev)
            and ev["start_frame"] <= last["end_frame"] + max_gap_frames
        ):
            last["end_frame"] = max(last["end_frame"], ev["end_frame"])
            last["end_time"] = max(last["end_time"], ev["end_time"])
            last["confidence"] = max(last["confidence"], ev["confidence"])
        else:
            merged.append(ev)
    return merged


def decide_final_class(fight_events, stalking_events):
    predictions = []
    predictions.extend(fight_events)
    predictions.extend(stalking_events)
    predictions = sorted(predictions, key=lambda x: x.get("confidence", 0), reverse=True)

    has_fight = len(fight_events) > 0
    has_stalking = len(stalking_events) > 0

    if has_fight:
        cls = "fight"
        main = max(fight_events, key=lambda x: x.get("confidence", 0))
    elif has_stalking:
        cls = "stalking"
        main = max(stalking_events, key=lambda x: x.get("confidence", 0))
    else:
        return {
            "detected": False,
            "final_class": "other",
            "main_event": "other",
            "confidence": 0.0,
            "predictions": [],
            "reason": "싸움 또는 스토킹 조건을 만족하는 후보가 없어 other로 분류함."
        }

    return {
        "detected": True,
        "final_class": cls,
        "main_event": cls,
        "confidence": main.get("confidence", 0),
        "predictions": predictions,
        "reason": f"{cls} 후보 이벤트가 탐지되어 최종 {cls}로 분류함."
    }

In [9]:
# ==============================
# 8. 이벤트 이미지/클립 저장
# ==============================
def find_video_for_cf2_data(cf2_data, fallback_video_path=None):
    candidates = []
    if cf2_data.get("video_path"):
        candidates.append(Path(cf2_data["video_path"]))
    if cf2_data.get("cf2_source_video"):
        candidates.append(Path(cf2_data["cf2_source_video"]))
    if fallback_video_path:
        candidates.append(Path(fallback_video_path))

    video_name = cf2_data.get("video_name")
    if video_name:
        candidates.extend(SAMPLE_DIR.rglob(video_name))

    for p in candidates:
        if p.exists():
            return p
    return None


def save_event_start_frame(video_path, event, out_path):
    if video_path is None or not Path(video_path).exists():
        return None
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    start_frame = int(event.get("start_frame", 0))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    cv2.imwrite(str(out_path), frame)
    return str(out_path)


def save_event_clip(video_path, event, out_path, margin_sec=1.0):
    if video_path is None or not Path(video_path).exists():
        return None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if not fps or fps <= 0:
        fps = 30.0

    start_frame = max(0, int(event.get("start_frame", 0) - margin_sec * fps))
    end_frame = min(total_frames - 1, int(event.get("end_frame", start_frame) + margin_sec * fps))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    cur = start_frame
    while cur <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break
        writer.write(frame)
        cur += 1

    writer.release()
    cap.release()
    return str(out_path)


def save_artifacts_for_events(video_path, video_stem, events):
    saved = []
    for idx, ev in enumerate(events, start=1):
        event_class = ev.get("event_class", "event")
        base = f"{video_stem}_{idx:02d}_{event_class}_{ev.get('start_frame')}_{ev.get('end_frame')}"
        frame_path = EVENT_FRAME_DIR / f"{base}.jpg"
        clip_path = EVENT_CLIP_DIR / f"{base}.mp4"

        frame_saved = save_event_start_frame(video_path, ev, frame_path)
        clip_saved = save_event_clip(video_path, ev, clip_path)

        ev["artifact_frame"] = frame_saved
        ev["artifact_clip"] = clip_saved
        saved.append({
            "event_class": event_class,
            "frame": frame_saved,
            "clip": clip_saved
        })
    return saved

In [10]:
# ==============================
# 9. 영상 하나 처리 함수
# ==============================
def process_one_video(video_path, cf2_json_path):
    video_path = Path(video_path)
    cf2_json_path = Path(cf2_json_path)
    cf2_data = load_json(cf2_json_path)

    tracks = assign_pseudo_tracks(cf2_data)
    frame_map = build_frame_map(tracks)

    if len(tracks) < 2:
        fight_events = []
        stalking_events = []
        final_prediction = {
            "detected": False,
            "final_class": "other",
            "main_event": "other",
            "confidence": 0.0,
            "predictions": [],
            "reason": "유효 person track이 2개 미만이라 행동 관계 분석이 어려워 other로 분류함."
        }
    else:
        fight_events = detect_fight(tracks, frame_map)
        stalking_events = detect_stalking(tracks, frame_map)
        final_prediction = decide_final_class(fight_events, stalking_events)

    all_events = []
    all_events.extend(fight_events)
    all_events.extend(stalking_events)
    all_events = sorted(all_events, key=lambda x: (x.get("start_frame", 0), x.get("event_class", "")))

    artifacts = save_artifacts_for_events(video_path, cf2_json_path.stem, all_events)

    output = {
        "task": "bd3_fight_stalking_other_classification",
        "video_name": cf2_data.get("video_name", video_path.name),
        "video_path": str(video_path),
        "input_json": str(cf2_json_path),
        "output_dir": str(OUTPUT_DIR),
        "track_count": len(tracks),
        "track_summaries": summarize_tracks(tracks),
        "fight_events": fight_events,
        "stalking_events": stalking_events,
        "final_prediction": final_prediction,
        "artifacts": artifacts,
        "decision_rule": {
            "fight": "손/발 근사 ROI가 상대 bbox와 겹치거나 가까워지는 상황이 3초 내 2회 이상 반복되면 fight 후보.",
            "stalking": "follower가 target의 뒤쪽에서 비슷한 방향으로 일정 거리 범위를 유지하며 이동하면 stalking 후보.",
            "other": "fight/stalking 조건을 만족하지 않으면 other."
        },
        "parameters": {
            "frame_stride": FRAME_STRIDE,
            "conf_th": CONF_TH,
            "min_track_length": MIN_TRACK_LENGTH,
            "fight_near_th": FIGHT_NEAR_TH,
            "fight_min_contact_count": FIGHT_MIN_CONTACT_COUNT,
            "fight_window_sec": FIGHT_WINDOW_SEC,
            "stalking_min_dist": STALKING_MIN_DIST,
            "stalking_max_dist": STALKING_MAX_DIST,
            "stalking_dir_sim_th": STALKING_DIR_SIM_TH,
            "stalking_window_sec": STALKING_WINDOW_SEC,
            "stalking_min_ratio": STALKING_MIN_RATIO,
        }
    }

    out_json = OUTPUT_DIR / f"{cf2_json_path.stem}_bd3_prediction.json"
    save_json(output, out_json)
    return output, out_json

In [11]:
# ==============================
# 10. sample 전체 영상 bd3 일괄 처리
# ==============================
results = []
failed = []

for row in cf2_generation_log:
    if row["status"] == "failed":
        failed.append({
            "video_path": row["video_path"],
            "cf2_json": row["cf2_json"],
            "stage": "cf2_generation",
            "error": row["error"]
        })
        continue

    video_path = Path(row["video_path"])
    cf2_json = Path(row["cf2_json"])

    try:
        print(f"\n[BD3 처리] {video_path.name}")
        output, out_json = process_one_video(video_path, cf2_json)
        fp = output["final_prediction"]

        results.append({
            "video_name": output.get("video_name"),
            "video_path": str(video_path),
            "cf2_json": str(cf2_json),
            "prediction_json": str(out_json),
            "final_class": fp.get("final_class"),
            "main_event": fp.get("main_event"),
            "confidence": fp.get("confidence"),
            "detected": fp.get("detected"),
            "fight_count": len(output.get("fight_events", [])),
            "stalking_count": len(output.get("stalking_events", [])),
            "track_count": output.get("track_count", 0),
        })
        print(f"  -> class={fp.get('final_class')} conf={fp.get('confidence')} fight={len(output.get('fight_events', []))} stalking={len(output.get('stalking_events', []))}")
    except Exception as e:
        failed.append({
            "video_path": str(video_path),
            "cf2_json": str(cf2_json),
            "stage": "bd3_processing",
            "error": repr(e)
        })
        print(f"[BD3 실패] {video_path.name}: {e}")

summary = {
    "task": "bd3_fight_stalking_other_batch",
    "sample_dir": str(SAMPLE_DIR),
    "cf2_dir": str(CF2_DIR),
    "output_dir": str(OUTPUT_DIR),
    "total_videos": len(video_paths),
    "success_count": len(results),
    "failed_count": len(failed),
    "fight_count": sum(1 for r in results if r["final_class"] == "fight"),
    "stalking_count": sum(1 for r in results if r["final_class"] == "stalking"),
    "other_count": sum(1 for r in results if r["final_class"] == "other"),
    "results": results,
    "failed": failed,
}

summary_path = OUTPUT_DIR / "bd3_batch_summary.json"
save_json(summary, summary_path)

print("\n================================")
print("bd3 전체 처리 완료")
print("================================")
print("전체 영상:", summary["total_videos"])
print("성공:", summary["success_count"])
print("실패:", summary["failed_count"])
print("fight:", summary["fight_count"])
print("stalking:", summary["stalking_count"])
print("other:", summary["other_count"])
print("summary 저장:", summary_path)


bd3 전체 처리 완료
전체 영상: 0
성공: 0
실패: 0
fight: 0
stalking: 0
other: 0
summary 저장: C:\Users\우성한\outputs\prediction\bd3\bd3_batch_summary.json


In [12]:
# ==============================
# 11. 최종 분류 파일 목록 출력
# ==============================
summary_path = OUTPUT_DIR / "bd3_batch_summary.json"
summary = load_json(summary_path)
results = summary.get("results", [])

for cls in ["fight", "stalking", "other"]:
    rows = [r for r in results if r.get("final_class") == cls]
    print("\n================================")
    print(f"{cls.upper()} : {len(rows)}개")
    print("================================")
    for r in rows:
        print(f"- {r.get('video_name')} | conf={r.get('confidence')} | fight={r.get('fight_count')} | stalking={r.get('stalking_count')}")
        print(f"  video: {r.get('video_path')}")
        print(f"  json : {r.get('prediction_json')}")


FIGHT : 0개

STALKING : 0개

OTHER : 0개


In [13]:
# ==============================
# 12. 이벤트 시간 구간 상세 출력
# ==============================
summary = load_json(OUTPUT_DIR / "bd3_batch_summary.json")

for r in summary.get("results", []):
    pred_path = Path(r["prediction_json"])
    data = load_json(pred_path)
    fight_events = data.get("fight_events", [])
    stalking_events = data.get("stalking_events", [])

    if not fight_events and not stalking_events:
        continue

    print("\n================================")
    print(data.get("video_name"))
    print("final_class:", data.get("final_prediction", {}).get("final_class"))
    print("prediction_json:", pred_path)
    print("================================")

    for ev in fight_events:
        print(
            f"[FIGHT] {ev.get('start_time')} ~ {ev.get('end_time')} sec | "
            f"frame {ev.get('start_frame')}~{ev.get('end_frame')} | "
            f"conf={ev.get('confidence')} | clip={ev.get('artifact_clip')}"
        )

    for ev in stalking_events:
        print(
            f"[STALKING] {ev.get('start_time')} ~ {ev.get('end_time')} sec | "
            f"frame {ev.get('start_frame')}~{ev.get('end_frame')} | "
            f"conf={ev.get('confidence')} | clip={ev.get('artifact_clip')}"
        )

In [14]:
# ==============================
# 13. 판정 결과를 CSV로 저장
# ==============================
import csv

csv_path = OUTPUT_DIR / "bd3_classification_results.csv"

with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "video_name", "final_class", "confidence", "fight_count", "stalking_count",
        "track_count", "video_path", "cf2_json", "prediction_json"
    ])
    writer.writeheader()
    for r in results:
        writer.writerow({
            "video_name": r.get("video_name"),
            "final_class": r.get("final_class"),
            "confidence": r.get("confidence"),
            "fight_count": r.get("fight_count"),
            "stalking_count": r.get("stalking_count"),
            "track_count": r.get("track_count"),
            "video_path": r.get("video_path"),
            "cf2_json": r.get("cf2_json"),
            "prediction_json": r.get("prediction_json"),
        })

print("CSV 저장:", csv_path)

CSV 저장: C:\Users\우성한\outputs\prediction\bd3\bd3_classification_results.csv


In [15]:
from pathlib import Path

SAMPLE_DIR = Path(r"C:\Users\우성한\sample")
CF2_DIR = Path(r"C:\Users\우성한\outputs\prediction\cf2")
BD3_DIR = Path(r"C:\Users\우성한\outputs\prediction\bd3")

print("SAMPLE_DIR exists:", SAMPLE_DIR.exists())
print("CF2_DIR exists:", CF2_DIR.exists())
print("BD3_DIR exists:", BD3_DIR.exists())

if SAMPLE_DIR.exists():
    print("\nsample 내부 상위 항목:")
    for p in list(SAMPLE_DIR.iterdir())[:20]:
        print(p)

SAMPLE_DIR exists: False
CF2_DIR exists: False
BD3_DIR exists: True


In [16]:
from pathlib import Path

VIDEO_EXTS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv",
    ".MP4", ".AVI", ".MOV", ".MKV", ".WMV"
}

video_paths = [
    p for p in SAMPLE_DIR.rglob("*")
    if p.is_file() and p.suffix in VIDEO_EXTS
]

print("찾은 영상 수:", len(video_paths))

for p in video_paths[:20]:
    print(p)

if len(video_paths) == 0:
    print("\n영상이 0개입니다.")
    print("확인할 것:")
    print("1. SAMPLE_DIR 경로가 실제 sample 폴더인지")
    print("2. 영상이 sample 폴더의 하위 폴더 안에 있는지")
    print("3. 영상 확장자가 mp4/avi/mov/mkv/wmv가 맞는지")

찾은 영상 수: 0

영상이 0개입니다.
확인할 것:
1. SAMPLE_DIR 경로가 실제 sample 폴더인지
2. 영상이 sample 폴더의 하위 폴더 안에 있는지
3. 영상 확장자가 mp4/avi/mov/mkv/wmv가 맞는지
